In [5]:
import pandas as pd
import numpy as np

dbox   = pd.read_csv("fractal_dimensions.csv")
report = pd.read_csv("sk_all50_report.csv")

# All tickers in universe
ALL_TICKERS = [
    "AAPL","MSFT","GOOG","AMZN","NVDA","META","TSLA","NFLX","AMD","INTC",
    "JPM","BAC","GS","MS","WFC","BLK","C","AXP","V","MA",
    "JNJ","PFE","UNH","MRK","ABBV","PG","KO","PEP","WMT","COST",
    "XOM","CVX","COP","BA","CAT","GE","HON","LMT","RTX","UPS",
    "GC=F","SI=F","CL=F","BTC-USD","ETH-USD",
    "EWJ","EWZ","FXI","EWG","EWU",
]

Q_VALUES = [10, 50, 75, 100, 150, 200]  # skip q=1 (delta~0)

# ── D_k from report: mean_sk_k2 averaged across windows, per (ticker, t)
# D_k = -mean_sk_k2
dk_base = (report
           .groupby(["ticker", "t"])["mean_sk_k2"]
           .mean()
           .reset_index()
           .rename(columns={"ticker": "Ticker", "mean_sk_k2": "sk_k2"}))
dk_base["D_k"] = -dk_base["sk_k2"]

# ── For tickers with Gamma=0, D_k=0 at all horizons
# Build a complete grid: every ticker × every t in report
all_t_values = sorted(dk_base["t"].unique())

full_grid = pd.MultiIndex.from_product(
    [ALL_TICKERS, all_t_values], names=["Ticker", "t"]
).to_frame(index=False)

dk_full = full_grid.merge(dk_base[["Ticker","t","D_k"]],
                          on=["Ticker","t"], how="left")
dk_full["D_k"] = dk_full["D_k"].fillna(0.0)  # unflagged => D_k=0

# ── Build lambda_implied for each (ticker, q, t)
rows = []
for q in Q_VALUES:
    dbox_q = dbox[dbox["q"] == q][["Ticker","D_box"]].copy()
    dbox_q["delta"] = dbox_q["D_box"] - 1.0

    merged = dk_full.merge(dbox_q, on="Ticker", how="left")

    for _, r in merged.iterrows():
        delta = r["delta"]
        D_k   = r["D_k"]
        lam   = (D_k * (q**2) / delta
                 if (not np.isnan(delta) and abs(delta) > 1e-6)
                 else np.nan)
        rows.append({
            "Ticker":         r["Ticker"],
            "q":              q,
            "t":              r["t"],
            "D_k":            D_k,
            "D_box":          r["D_box"],
            "delta":          delta,
            "lambda_implied": lam,
        })

result = pd.DataFrame(rows)

# ── Per-ticker summary: mean/std/median/min/max across all (q,t)
summary = (result
           .groupby("Ticker")["lambda_implied"]
           .agg(["mean","std","median","min","max"])
           .reset_index()
           .rename(columns={
               "mean":   "lambda_mean",
               "std":    "lambda_std",
               "median": "lambda_median",
               "min":    "lambda_min",
               "max":    "lambda_max",
           })
           .sort_values("lambda_mean"))

print("── Per-ticker lambda_implied summary (q>1, all t) ──")
print(summary.to_string(index=False))

# ── Per (ticker, q): mean across t
pivot_q = (result
           .groupby(["Ticker","q"])["lambda_implied"]
           .mean()
           .reset_index()
           .pivot(index="Ticker", columns="q", values="lambda_implied")
           .reset_index())
pivot_q.columns = ["Ticker"] + [f"lam_q{int(c)}" for c in pivot_q.columns[1:]]

print("\n── Per-ticker lambda_implied by q (mean across t) ──")
print(pivot_q.to_string(index=False))

# ── Per (ticker, t): mean across q  — no NaN for zero-Gamma tickers
pivot_t = (result
           .groupby(["Ticker","t"])["lambda_implied"]
           .mean()
           .reset_index()
           .pivot(index="Ticker", columns="t", values="lambda_implied")
           .reset_index())

print("\n── Per-ticker lambda_implied by t (mean across q) ──")
print(pivot_t.to_string(index=False))

result.to_csv("lambda_implied_full.csv", index=False)
summary.to_csv("lambda_implied_summary.csv", index=False)
print("\nSaved: lambda_implied_full.csv, lambda_implied_summary.csv")

── Per-ticker lambda_implied summary (q>1, all t) ──
 Ticker  lambda_mean  lambda_std  lambda_median   lambda_min  lambda_max
    AMD  -118.001152  432.868131       0.567045 -2572.757320   68.975520
    CAT     0.000000    0.000000       0.000000     0.000000    0.000000
   GC=F     0.000000    0.000000       0.000000     0.000000    0.000000
    EWJ     0.000000    0.000000       0.000000     0.000000    0.000000
    JNJ     0.000000    0.000000       0.000000     0.000000    0.000000
   GOOG     0.000000    0.000000       0.000000     0.000000    0.000000
   SI=F     0.000000    0.000000       0.000000     0.000000    0.000000
    PFE     0.000000    0.000000       0.000000     0.000000    0.000000
   COST     0.485406    1.485152       0.000000     0.000000    7.318046
    XOM     3.807490   10.667100       0.000000     0.000000   53.543346
    WMT     4.187443   12.698227       0.000000     0.000000   61.027369
    EWG     4.621041   14.428537       0.000000     0.000000   71.50529

In [ ]:
"""
s_k / D_k Analysis — All 50 Benchmark Tickers, Multiple Windows
================================================================
Runs the single-ticker multi-window analysis across every ticker in the
benchmark universe. Downloads each ticker once (IPO → END_DATE), then
analyses each window independently.

Outputs
-------
sk_plots_all50/<TICKER>/          — per-ticker 4-panel PNG per window
sk_all50_report.csv               — full flagged-horizon table (all tickers)
sk_all50_summary.csv              — one row per (ticker, window): dk_signal,
                                    n_flagged, flagged horizons
sk_all50_heatmap.png              — heatmap: tickers × windows, dk_signal
"""

import os, warnings, time
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from scipy.stats import spearmanr
import yfinance as yf
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════
END_DATE     = "2026-04-11"
WINDOW_SIZES = [500, 1000, 2000, 3000, 5000, 7500, 10000, 15000]

SECTORS = {
    "Tech":                  ["AAPL","MSFT","GOOG","AMZN","NVDA","META","TSLA","NFLX","AMD","INTC"],
    "Financials":            ["JPM","BAC","GS","MS","WFC","BLK","C","AXP","V","MA"],
    "Healthcare & Consumer": ["JNJ","PFE","UNH","MRK","ABBV","PG","KO","PEP","WMT","COST"],
    "Energy & Industrials":  ["XOM","CVX","COP","BA","CAT","GE","HON","LMT","RTX","UPS"],
    "Commodities & Crypto":  ["GC=F","SI=F","CL=F","BTC-USD","ETH-USD"],
    "Intl ETFs":             ["EWJ","EWZ","FXI","EWG","EWU"],
}
ALL_TICKERS  = [t for ts in SECTORS.values() for t in ts]
SECTOR_OF    = {t: s for s, ts in SECTORS.items() for t in ts}

K_VALUES = [0.5, 1.0, 2.0, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]

CF_MIN_MAG                = 0.30
RELATIVE_CHANGE_THRESHOLD = 0.05
ABSOLUTE_CHANGE_THRESHOLD = 5e-5
MIN_VALID_K               = 4
VALIDITY_RATIO            = 0.60
MIN_ABS_MEAN              = 1e-5
CLIP_IQR                  = 3.0

_ALL_CANDIDATE_HORIZONS = [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500, 2000]

OUT_DIR  = "sk_plots_all50"
os.makedirs(OUT_DIR, exist_ok=True)


# ══════════════════════════════════════════════════════════════════
# DERIVED PARAMS
# ══════════════════════════════════════════════════════════════════
def derive_params(window_days):
    max_horizon = window_days // 2
    horizons    = [t for t in _ALL_CANDIDATE_HORIZONS if t <= max_horizon]
    n_origins   = (window_days - 1) - max(horizons)
    return horizons, n_origins


# ══════════════════════════════════════════════════════════════════
# DATA
# ══════════════════════════════════════════════════════════════════
def fetch_ticker(ticker):
    """Download full history. Returns pd.Series of Close prices or None."""
    try:
        df = yf.download(ticker, start="1900-01-01", end=END_DATE,
                         progress=False, auto_adjust=True)
        if df.empty:
            return None
        prices = df["Close"].squeeze().dropna()
        return prices if len(prices) >= 100 else None
    except Exception:
        return None


def slice_window(prices, window_days):
    if len(prices) < window_days:
        return None, None, None
    sliced = prices.iloc[-window_days:]
    return (np.diff(np.log(sliced.values)),
            sliced.index[0].strftime("%Y-%m-%d"),
            sliced.index[-1].strftime("%Y-%m-%d"))


# ══════════════════════════════════════════════════════════════════
# CORE COMPUTATION  (identical to original)
# ══════════════════════════════════════════════════════════════════
def build_ensemble(log_returns, horizons, n_origins):
    displacements = {}
    for t in horizons:
        valid = [log_returns[i + t] - log_returns[i]
                 for i in range(n_origins) if i + t < len(log_returns)]
        displacements[t] = np.array(valid) if valid else np.array([])
    return displacements


def compute_sk(displacements_t, k, t):
    n = len(displacements_t)
    if n < 5:
        return np.nan, n, np.nan
    char_fn   = np.mean(np.exp(1j * k * displacements_t))
    magnitude = np.abs(char_fn)
    if magnitude < 1e-15:
        return np.nan, n, magnitude
    return (1.0 / t) * np.log(magnitude), n, magnitude


def compute_dk(sk, k):
    if np.isnan(sk) or k == 0:
        return np.nan
    return -sk / (k ** 2)


def run_analysis(displacements, horizons):
    results = {}
    for k in K_VALUES:
        results[k] = {}
        for t in horizons:
            sk, n, mag = compute_sk(displacements[t], k, t)
            dk         = compute_dk(sk, k)
            sk_over_k2 = sk / k**2 if not np.isnan(sk) else np.nan
            results[k][t] = (sk, dk, sk_over_k2, n, mag)
    return results


def detect_non_constancy(results, horizons):
    flagged   = []
    n_k_total = len(K_VALUES)
    for t in horizons:
        k_arr   = np.array(K_VALUES)
        sk2_arr = np.array([results[k][t][2] for k in K_VALUES])
        mag_arr = np.array([results[k][t][4] for k in K_VALUES])

        reliable       = (~np.isnan(sk2_arr)) & (~np.isnan(mag_arr)) & (mag_arr >= CF_MIN_MAG)
        vals           = sk2_arr[reliable]
        k_rel          = k_arr[reliable]
        n_rel          = int(reliable.sum())
        coverage_ratio = n_rel / n_k_total

        if n_rel < MIN_VALID_K or coverage_ratio < VALIDITY_RATIO:
            continue

        min_val    = np.min(vals)
        max_val    = np.max(vals)
        abs_change = max_val - min_val
        mean_abs   = np.mean(np.abs(vals))

        if mean_abs > MIN_ABS_MEAN:
            rel_change  = abs_change / mean_abs
            change_type = "relative"
        else:
            rel_change  = abs_change / ABSOLUTE_CHANGE_THRESHOLD
            change_type = "absolute"

        spearman_r, spearman_p = np.nan, np.nan
        if n_rel >= 4:
            spearman_r, spearman_p = spearmanr(k_rel, vals)

        should_flag  = False
        reason_parts = []
        if change_type == "relative" and rel_change > RELATIVE_CHANGE_THRESHOLD:
            should_flag = True
            reason_parts.append(f"rel Δ={rel_change:.1%} > {RELATIVE_CHANGE_THRESHOLD:.0%}")
        elif change_type == "absolute" and abs_change > ABSOLUTE_CHANGE_THRESHOLD:
            should_flag = True
            reason_parts.append(f"abs Δ={abs_change:.2e} > {ABSOLUTE_CHANGE_THRESHOLD:.0e}")

        if should_flag:
            direction = "increasing" if max_val > min_val else "decreasing"
            reason_parts.append(f"({direction} across k)")
            flagged.append({
                "t":          t,
                "n_reliable": n_rel,
                "coverage":   coverage_ratio,
                "mean_sk_k2": float(np.mean(vals)),
                "std_sk_k2":  float(np.std(vals, ddof=1) if n_rel > 1 else 0.0),
                "min_val":    min_val,
                "max_val":    max_val,
                "abs_change": abs_change,
                "rel_change": rel_change if change_type == "relative" else np.nan,
                "spearman_r": spearman_r,
                "spearman_p": spearman_p,
                "reason":     " & ".join(reason_parts),
            })
    return flagged


def dk_signal_from_flagged(flagged, horizons):
    """
    Aggregate non-constancy score: mean rel_change over short-term horizons
    (t <= 20), capped at 5. Returns 0.0 if nothing flagged at short horizons.
    Mirrors the logic used in the benchmark scripts.
    """
    short_vals = []
    for fd in flagged:
        if fd["t"] <= 20:
            rc = fd["rel_change"] if not np.isnan(fd["rel_change"]) else fd["abs_change"] / ABSOLUTE_CHANGE_THRESHOLD
            short_vals.append(float(np.clip(rc, 0., 5.)))
    return float(np.mean(short_vals)) if short_vals else 0.0


# ══════════════════════════════════════════════════════════════════
# PLOT HELPERS  (unchanged from original)
# ══════════════════════════════════════════════════════════════════
def make_k_colors(k_values, cmap_name="turbo"):
    try:    cmap = plt.colormaps[cmap_name]
    except: cmap = cm.get_cmap(cmap_name, max(len(k_values), 2))
    return [cmap(p) for p in np.linspace(0.05, 0.95, len(k_values))]

def make_h_colors(horizons, cmap_name="plasma"):
    try:    cmap = plt.colormaps[cmap_name]
    except: cmap = cm.get_cmap(cmap_name, max(len(horizons), 2))
    return [cmap(p) for p in np.linspace(0.1, 0.9, len(horizons))]

def robust_ylim(values_list, iqr_mult=CLIP_IQR, pad=0.08):
    arr = np.array([v for v in values_list if not np.isnan(v)])
    if len(arr) == 0: return -1, 1
    q1, q3 = np.percentile(arr, [25, 75])
    iqr = q3 - q1
    if iqr == 0: iqr = max(abs(np.ptp(arr)), 1e-10)
    lo, hi = q1 - iqr_mult * iqr, q3 + iqr_mult * iqr
    span = hi - lo if hi > lo else max(abs(hi), 1e-10)
    return lo - pad * span, hi + pad * span

def apply_robust_ylim(ax, values_list):
    ymin, ymax = robust_ylim(values_list)
    ax.set_ylim(ymin, ymax)
    return ymin, ymax

def annotate_cf(ax, x, y, mag, color):
    if not np.isnan(mag) and not np.isnan(y) and mag < 0.50:
        ax.annotate(f"|CF|={mag:.2f}", xy=(x, y), xytext=(0, 8),
                    textcoords="offset points", ha="center", fontsize=6,
                    color=color, alpha=0.85,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white",
                              ec=color, lw=0.5, alpha=0.7))

def place_legend(ax, n_items, fontsize=25):
    ax.legend(fontsize=max(fontsize - 2, 6), loc="upper left",
              bbox_to_anchor=(1.01, 1), borderaxespad=0,
              ncol=max(1, n_items // 12 + 1), framealpha=0.9)

def _add_clipped_markers(ax, xs, ys, color, ymin, ymax):
    for x, y in zip(xs, ys):
        if np.isnan(y): continue
        if y > ymax: ax.plot(x, ymax * 0.98, marker="^", color=color, ms=6, zorder=5, clip_on=False)
        elif y < ymin: ax.plot(x, ymin * 0.98, marker="v", color=color, ms=6, zorder=5, clip_on=False)


# ══════════════════════════════════════════════════════════════════
# PER-WINDOW PLOT  (identical 4-panel layout as original)
# ══════════════════════════════════════════════════════════════════
def plot_results(results, horizons, ticker, window_days, window_start,
                 window_end, flagged_ts, plot_dir):
    plt.rcParams.update({
        "axes.facecolor": "white", "figure.facecolor": "white",
        "axes.edgecolor": "#cccccc", "axes.labelcolor": "#111111",
        "xtick.color": "#333333", "ytick.color": "#333333",
        "text.color": "#111111", "grid.color": "#e5e5e5",
        "grid.linewidth": 0.6, "legend.framealpha": 0.9,
        "legend.edgecolor": "#dddddd",
    })

    n_k      = len(K_VALUES)
    k_colors = make_k_colors(K_VALUES)
    h_colors = make_h_colors(horizons)
    lw = max(0.8, 2.2 - 0.07 * n_k)
    ms = max(2.5, 5.0 - 0.15 * n_k)

    n_rep    = results[K_VALUES[0]][horizons[0]][3]
    flag_str = (f"⚑ Non-constant at t={[f['t'] for f in flagged_ts]}"
                if flagged_ts else "✓ Constant across all t")
    window_label = f"Window: {window_days}d  ({window_start} → {window_end})"

    fig = plt.figure(figsize=(20, 28), facecolor="white")
    '''
    fig.suptitle(
        f"{ticker}  |  {window_label}  |  N={n_rep} origins\n{flag_str}",
        fontsize=11, color="#111111", y=0.998, fontfamily="serif",
    )'''
    gs  = gridspec.GridSpec(4, 1, figure=fig, hspace=0.55)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])
    ax4 = fig.add_subplot(gs[3])

    for ax in [ax1, ax2, ax3, ax4]:
        ax.set_facecolor("white"); ax.grid(True, lw=0.6)
        ax.axhline(0, color="#aaaaaa", lw=0.9, ls="--")
        for spine in ax.spines.values(): spine.set_edgecolor("#cccccc")
        for fd in flagged_ts:
            ax.axvline(fd["t"], color="#e63946", lw=0.8, ls=":", alpha=0.5)

    # Panel 1: s_k/k² vs t
    all_p1 = [results[k][t][2] for k in K_VALUES for t in horizons if not np.isnan(results[k][t][2])]
    ymin1, ymax1 = apply_robust_ylim(ax1, all_p1)
    for i, k in enumerate(K_VALUES):
        ts   = [t for t in horizons if not np.isnan(results[k][t][2])]
        vals = [results[k][t][2] for t in ts]
        mags = [results[k][t][4] for t in ts]
        ax1.plot(ts, vals, color=k_colors[i], lw=lw, marker="o", ms=ms, label=f"k={k}")
        for t, v, mag in zip(ts, vals, mags):
            annotate_cf(ax1, t, v, mag, k_colors[i])
        _add_clipped_markers(ax1, ts, vals, k_colors[i], ymin1, ymax1)
    ax1.set_xlabel(r"Horizon $t$ (days)", fontsize=20)
    ax1.set_ylabel(r"$s_k(t)/k^2$  (days$^{-1}$)", fontsize=20)
    #ax1.set_title(r"$s_k/k^2$ vs $t$" + f"\n[IQR×{CLIP_IQR:.0f} clip]", fontsize=9, fontweight="bold")
    place_legend(ax1, n_k)

    # Panel 2: s_k/k² vs k
    ax2.set_xlim(K_VALUES[0], K_VALUES[-1])
    for j, t in enumerate(horizons):
        rel_mask = np.array([
            not np.isnan(results[k][t][2]) and
            not np.isnan(results[k][t][4]) and
            results[k][t][4] >= CF_MIN_MAG
            for k in K_VALUES])
        vals = np.array([results[k][t][2] for k in K_VALUES])
        mags = np.array([results[k][t][4] for k in K_VALUES])
        ns   = results[K_VALUES[0]][t][3]
        n_display = f"{ns/1000:.1f}k" if ns >= 1000 else str(ns)
        fd   = next((f for f in flagged_ts if f["t"] == t), None)
        lbl  = f"t={t}d (N={n_display})" + (" ⚑" if fd else "")
        ax2.plot(K_VALUES, vals, color=h_colors[j], lw=2, marker="o", ms=5, label=lbl)
        if rel_mask.sum() >= 2:
            ax2.axhline(np.mean(vals[rel_mask]), color=h_colors[j], lw=0.8, ls="--", alpha=0.4)
        for ki, k in enumerate(K_VALUES):
            if not rel_mask[ki] and not np.isnan(vals[ki]):
                ax2.plot(k, vals[ki], "x", color=h_colors[j], ms=8, markeredgewidth=1.5, alpha=0.6)
            annotate_cf(ax2, k, vals[ki], mags[ki], h_colors[j])
    ax2.set_xlabel(r"Wave number $k$", fontsize=20)
    ax2.set_ylabel(r"$s_k(t)/k^2$  (days$^{-1}$)", fontsize=20)
    place_legend(ax2, len(horizons))

    # Panel 3: raw s_k vs t
    all_p3 = [results[k][t][0] for k in K_VALUES for t in horizons if not np.isnan(results[k][t][0])]
    ymin3, ymax3 = apply_robust_ylim(ax3, all_p3)
    for i, k in enumerate(K_VALUES):
        ts   = [t for t in horizons if not np.isnan(results[k][t][0])]
        vals = [results[k][t][0] for t in ts]
        mags = [results[k][t][4] for t in ts]
        ax3.plot(ts, vals, color=k_colors[i], lw=lw, marker="o", ms=ms, label=f"k={k}")
        for t, v, mag in zip(ts, vals, mags):
            annotate_cf(ax3, t, v, mag, k_colors[i])
        _add_clipped_markers(ax3, ts, vals, k_colors[i], ymin3, ymax3)
    ax3.set_xlabel(r"Horizon $t$ (days)", fontsize=20)
    ax3.set_ylabel(r"$s_k(t)$  (days$^{-1}$)", fontsize=20)
    '''ax3.set_title(r"$s_k$ vs $t$ (raw)" + f"\n[IQR×{CLIP_IQR:.0f} clip]", fontsize=9, fontweight="bold")'''
    place_legend(ax3, n_k)

    # Panel 4: raw s_k vs k
    ax4.set_xlim(K_VALUES[0], K_VALUES[-1])
    for j, t in enumerate(horizons):
        vals = np.array([results[k][t][0] for k in K_VALUES])
        mags = np.array([results[k][t][4] for k in K_VALUES])
        ns   = results[K_VALUES[0]][t][3]
        n_display = f"{ns/1000:.1f}k" if ns >= 1000 else str(ns)
        ax4.plot(K_VALUES, vals, color=h_colors[j], lw=2, marker="o", ms=5, label=f"t={t}d (N={n_display})")
        for k, v, mag in zip(K_VALUES, vals, mags):
            annotate_cf(ax4, k, v, mag, h_colors[j])
    ax4.set_xlabel(r"Wave number $k$", fontsize=20)
    ax4.set_ylabel(r"$s_k(t)$  (days$^{-1}$)", fontsize=20)
    #ax4.set_title(r"$s_k$ vs $k$ (raw)", fontsize=10, fontweight="bold")
    place_legend(ax4, len(horizons))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    safe  = ticker.replace("=", "_").replace("-", "_")
    fname = os.path.join(plot_dir, f"{safe}_window{window_days}d.png")
    plt.savefig(fname, dpi=120, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return fname


# ══════════════════════════════════════════════════════════════════
# AGGREGATE HEATMAP  (new — tickers × windows, dk_signal)
# ══════════════════════════════════════════════════════════════════
def plot_heatmap(summary_rows, out_path):
    """
    Heatmap of dk_signal by (ticker, window).
    Rows = tickers (grouped by sector), columns = window sizes.
    Color = dk_signal magnitude; grey = insufficient data.
    """
    tickers_ordered = []
    sector_boundaries = {}
    y = 0
    for sector, tickers in SECTORS.items():
        sector_boundaries[sector] = (y, y + len(tickers))
        tickers_ordered.extend(tickers)
        y += len(tickers)

    windows = sorted(set(r["window_days"] for r in summary_rows))
    n_t, n_w = len(tickers_ordered), len(windows)

    # Build matrix
    dk_matrix = np.full((n_t, n_w), np.nan)
    idx_t = {t: i for i, t in enumerate(tickers_ordered)}
    idx_w = {w: j for j, w in enumerate(windows)}
    for row in summary_rows:
        ti = idx_t.get(row["ticker"])
        wi = idx_w.get(row["window_days"])
        if ti is not None and wi is not None:
            dk_matrix[ti, wi] = row["dk_signal"]

    fig, ax = plt.subplots(figsize=(max(8, n_w * 1.6), max(10, n_t * 0.35)))
    fig.patch.set_facecolor("white"); ax.set_facecolor("#f5f5f5")

    # Mask NaN as grey
    masked = np.ma.masked_invalid(dk_matrix)
    cmap   = plt.cm.YlOrRd.copy(); cmap.set_bad("#cccccc")
    vmax   = max(np.nanpercentile(dk_matrix[~np.isnan(dk_matrix)], 95), 0.1) if not np.all(np.isnan(dk_matrix)) else 1.0
    im     = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=0, vmax=vmax,
                       interpolation="nearest")

    # Cell annotations
    for i in range(n_t):
        for j in range(n_w):
            v = dk_matrix[i, j]
            if not np.isnan(v):
                txt_color = "white" if v > vmax * 0.6 else "#333333"
                ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                        fontsize=7.5, color=txt_color, fontweight="bold")
            else:
                ax.text(j, i, "n/a", ha="center", va="center",
                        fontsize=7, color="#888888")

    # Axes
    ax.set_xticks(range(n_w))
    ax.set_xticklabels([f"{w}d" for w in windows], fontsize=10)
    ax.set_yticks(range(n_t))
    ax.set_yticklabels(tickers_ordered, fontsize=9)
    ax.set_xlabel("Window size (trading days)", fontsize=11)
    '''ax.set_title(
        f"dk_signal by ticker × window  |  end={END_DATE}\n"
        f"Higher = more scale-dependent diffusion (CF non-constant)  |  grey=insufficient data",
        fontsize=11, fontweight="bold", pad=12
    )'''

    # Sector dividers and labels
    for sector, (y0, y1) in sector_boundaries.items():
        mid = (y0 + y1 - 1) / 2
        ax.axhline(y0 - 0.5, color="white", lw=2.5)
        ax.text(-0.7, mid, sector, ha="right", va="center", fontsize=8.5,
                color="#333333", fontweight="bold")

    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("dk_signal", fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Heatmap saved → {out_path}")


# ══════════════════════════════════════════════════════════════════
# PER-TICKER ANALYSIS
# ══════════════════════════════════════════════════════════════════
def analyse_ticker(ticker, all_prices):
    """
    Run the full multi-window analysis for one ticker.
    Returns (all_flag_rows, summary_rows).
    """
    prices = all_prices.get(ticker)
    if prices is None:
        return [], []

    ticker_dir = os.path.join(OUT_DIR, ticker.replace("=", "_").replace("-", "_"))
    os.makedirs(ticker_dir, exist_ok=True)

    all_flag_rows = []
    summary_rows  = []

    for window_days in WINDOW_SIZES:
        horizons, n_origins = derive_params(window_days)
        log_returns, window_start, window_end = slice_window(prices, window_days)

        if log_returns is None:
            continue

        needed = max(horizons) + n_origins
        if len(log_returns) < needed:
            continue

        displacements = build_ensemble(log_returns, horizons, n_origins)
        results       = run_analysis(displacements, horizons)
        flagged       = detect_non_constancy(results, horizons)
        dk_sig        = dk_signal_from_flagged(flagged, horizons)

        # Per-window plot
        try:
            plot_results(results, horizons, ticker, window_days,
                         window_start, window_end, flagged, ticker_dir)
        except Exception as e:
            print(f"    [WARN] Plot failed for {ticker} w={window_days}: {e}")

        # Collect full flag rows
        for fd in flagged:
            all_flag_rows.append({
                "ticker":       ticker,
                "sector":       SECTOR_OF.get(ticker, "Unknown"),
                "window_days":  window_days,
                "window_start": window_start,
                "window_end":   window_end,
                **fd,
            })

        # Summary row
        summary_rows.append({
            "ticker":          ticker,
            "sector":          SECTOR_OF.get(ticker, "Unknown"),
            "window_days":     window_days,
            "window_start":    window_start,
            "window_end":      window_end,
            "dk_signal":       dk_sig,
            "n_flagged":       len(flagged),
            "flagged_horizons": str([f["t"] for f in flagged]) if flagged else "[]",
            "n_available_days": len(prices),
        })

    return all_flag_rows, summary_rows


# ══════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()

    print("=" * 68)
    print("  s_k / D_k ANALYSIS — ALL 50 BENCHMARK TICKERS")
    print("=" * 68)
    print(f"  End date     : {END_DATE}")
    print(f"  Windows      : {WINDOW_SIZES}")
    print(f"  Tickers      : {len(ALL_TICKERS)}")
    print(f"  K values     : {K_VALUES}")
    print(f"  CF_MIN_MAG   : {CF_MIN_MAG}")
    print(f"  REL threshold: {RELATIVE_CHANGE_THRESHOLD:.0%}")
    print("=" * 68)

    # Print derived params table once
    print(f"\n  {'WINDOW':>8}  {'Log-ret':>8}  {'MaxHorizon':>12}  {'N_ORIGINS':>10}  HORIZONS")
    print(f"  {'─'*72}")
    for w in WINDOW_SIZES:
        h, n = derive_params(w)
        print(f"  {w:>8}  {w-1:>8}  {max(h):>12}  {n:>10}  {h}")
    print(f"  {'─'*72}\n")

    # ── Batch download all 50 tickers ─────────────────────────────
    print("Downloading all 50 tickers (full history)...")
    all_prices = {}
    for sector, tickers in SECTORS.items():
        print(f"\n  [{sector}]")
        for ticker in tickers:
            prices = fetch_ticker(ticker)
            if prices is not None:
                all_prices[ticker] = prices
                print(f"    {ticker:<12}  {len(prices):>5} days  "
                      f"{prices.index[0].strftime('%Y-%m-%d')} → {prices.index[-1].strftime('%Y-%m-%d')}")
            else:
                print(f"    {ticker:<12}  FAILED")

    print(f"\n  Downloaded: {len(all_prices)}/{len(ALL_TICKERS)} tickers\n")

    # ── Analyse all tickers ───────────────────────────────────────
    all_flag_rows_global = []
    all_summary_rows     = []

    for sector, tickers in SECTORS.items():
        print(f"\n{'─'*68}")
        print(f"  SECTOR: {sector}")
        print(f"{'─'*68}")

        for ticker in tickers:
            t_start = time.time()
            print(f"  {ticker:<12}", end=" ", flush=True)

            flag_rows, summary_rows = analyse_ticker(ticker, all_prices)
            all_flag_rows_global.extend(flag_rows)
            all_summary_rows.extend(summary_rows)

            # One-line per-ticker summary
            if summary_rows:
                parts = []
                for row in sorted(summary_rows, key=lambda r: r["window_days"]):
                    parts.append(f"{row['window_days']}d:dk={row['dk_signal']:.3f}(n={row['n_flagged']})")
                print("  " + "  ".join(parts) + f"  ({time.time()-t_start:.0f}s)")
            else:
                avail = len(all_prices.get(ticker, []))
                print(f"  SKIP (only {avail} days available)  ({time.time()-t_start:.0f}s)")

    # ── Save CSVs ─────────────────────────────────────────────────
    print(f"\n{'='*68}")
    print("  Saving outputs...")

    if all_flag_rows_global:
        df_full = pd.DataFrame(all_flag_rows_global)
        df_full = df_full.sort_values(["sector","ticker","window_days","t"]).reset_index(drop=True)
        df_full.to_csv("sk_all50_report.csv", index=False, float_format="%.6f")
        print(f"  Full report  → sk_all50_report.csv  ({len(df_full)} flagged rows)")
    else:
        print("  Full report  → (no flagged horizons found)")

    if all_summary_rows:
        df_sum = pd.DataFrame(all_summary_rows)
        df_sum = df_sum.sort_values(["sector","ticker","window_days"]).reset_index(drop=True)
        df_sum.to_csv("sk_all50_summary.csv", index=False, float_format="%.6f")
        print(f"  Summary      → sk_all50_summary.csv  ({len(df_sum)} rows)")

        # Print summary table sorted by mean dk_signal descending
        mean_dk = (df_sum.groupby("ticker")["dk_signal"]
                         .mean()
                         .sort_values(ascending=False)
                         .reset_index())
        mean_dk.columns = ["ticker", "mean_dk_signal"]
        mean_dk["sector"] = mean_dk["ticker"].map(SECTOR_OF)
        print(f"\n  {'Ticker':<12} {'Sector':<26} {'mean dk_signal':>14}")
        print(f"  {'─'*54}")
        for _, row in mean_dk.iterrows():
            bar = "█" * min(int(row["mean_dk_signal"] * 200), 30)
            print(f"  {row['ticker']:<12} {row['sector']:<26} {row['mean_dk_signal']:>14.4f}  {bar}")

        # Heatmap
        plot_heatmap(all_summary_rows, "sk_all50_heatmap.png")
    else:
        print("  Summary      → (no rows)")

    elapsed = time.time() - t0
    print(f"\n  Plots   → {OUT_DIR}/<TICKER>/")
    print(f"  Total wall time: {elapsed:.0f}s  ({elapsed/60:.1f} min)")
    print("=" * 68)
    print("Done.")


if __name__ == "__main__":
    main()

  s_k / D_k ANALYSIS — ALL 50 BENCHMARK TICKERS
  End date     : 2026-04-11
  Windows      : [500, 1000, 2000, 3000, 5000, 7500, 10000, 15000]
  Tickers      : 50
  K values     : [0.5, 1.0, 2.0, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
  CF_MIN_MAG   : 0.3
  REL threshold: 5%

    WINDOW   Log-ret    MaxHorizon   N_ORIGINS  HORIZONS
  ────────────────────────────────────────────────────────────────────────
       500       499           250         249  [1, 5, 10, 20, 40, 60, 120, 250]
      1000       999           500         499  [1, 5, 10, 20, 40, 60, 120, 250, 500]
      2000      1999          1000         999  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000]
      3000      2999          1500        1499  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500]
      5000      4999          2000        2999  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500, 2000]
      7500      7499          2000        5499  [1, 5, 10, 20, 40, 60, 120, 250, 500, 750, 1000, 1500, 2000]

In [2]:
"""
Φ(t; q) — Full Canvas SVG + Box-Counting Fractal Dimension
=============================================================
- Per-ticker subfolders inside OUT_DIR
- Saves SVG for each ticker × q  (unchanged)
- Saves crisp zoomed PNGs via matplotlib (data-space crop, 300 DPI)
  Zoom levels crop the data coordinates directly — no pixel resampling artifacts
"""

import numpy as np
import yfinance as yf
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
TICKERS = [
    # US Large Cap Tech
    "AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "AMD", "INTC",

    # US Financials
    "JPM", "BAC", "GS", "MS", "WFC",
    "BLK", "C", "AXP", "V", "MA",

    # US Healthcare & Consumer
    "JNJ", "PFE", "UNH", "MRK", "ABBV",
    "PG", "KO", "PEP", "WMT", "COST",

    # US Energy & Industrials
    "XOM", "CVX", "COP", "BA", "CAT",
    "GE", "HON", "LMT", "RTX", "UPS",

    # Commodities & Crypto
    "GC=F", "SI=F", "CL=F", "BTC-USD", "ETH-USD",

    # International ETFs
    "EWJ", "EWZ", "FXI", "EWG", "EWU",
]

START    = "2010-01-01"
END      = "2026-04-11"
Q_VALUES = [1, 10, 50, 75, 100, 150, 200]

OUT_DIR  = "phi_full_zoom"
os.makedirs(OUT_DIR, exist_ok=True)

# SVG canvas
W, H   = 8000, 8000
MARGIN = 100

# PNG settings
PNG_DPI     = 300    # 300 DPI × 10 in = 3000×3000 px output
PNG_SIZE_IN = 10     # inches
LINE_WIDTH  = 0.4    # pt — thin enough to reveal fine structure

# Zoom levels: fraction of the full data range shown, centred on the curve's
# robust midpoint. 1.0 = full extent, 0.0625 = 16× zoom into centre.
ZOOM_LEVELS = [
    ("z1_full", 1.00),
    ("z2_2x",   0.50),
    ("z3_4x",   0.25),
    ("z4_8x",   0.125),
    ("z5_16x",  0.0625),
]


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────

def get_returns(ticker):
    df  = yf.download(ticker, start=START, end=END, progress=False)
    px  = df["Close"].dropna().values.ravel().astype(float)
    px  = px[px > 0]
    ret = np.diff(np.log(px))
    return ret[np.isfinite(ret)]


# ─────────────────────────────────────────────────────────────────────────────
# Φ CURVE
# ─────────────────────────────────────────────────────────────────────────────

def phi(ret, q):
    return np.cumsum(np.exp(1j * q * ret))


# ─────────────────────────────────────────────────────────────────────────────
# BOX-COUNTING FRACTAL DIMENSION
# ─────────────────────────────────────────────────────────────────────────────

def box_counting_dimension(Z, n_scales=20, scale_min=None, scale_max=None):
    xr, yr = Z.real, Z.imag
    span   = max(xr.max() - xr.min(), yr.max() - yr.min(), 1e-12)

    xn = (xr - xr.min()) / span
    yn = (yr - yr.min()) / span

    if scale_max is None: scale_max = 0.20
    if scale_min is None: scale_min = 2.0 / len(Z)

    epsilons = np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales)

    Ns = []
    for eps in epsilons:
        ix       = np.floor(xn / eps).astype(np.int64)
        iy       = np.floor(yn / eps).astype(np.int64)
        occupied = len(set(zip(ix.tolist(), iy.tolist())))
        Ns.append(occupied)

    Ns      = np.array(Ns, dtype=float)
    log_eps = np.log(epsilons)
    log_N   = np.log(Ns)
    valid   = np.isfinite(log_N) & (Ns > 1)

    if valid.sum() < 4:
        return np.nan, epsilons, Ns, np.nan

    coeffs = np.polyfit(log_eps[valid], log_N[valid], 1)
    D_box  = -coeffs[0]

    fitted = np.polyval(coeffs, log_eps[valid])
    ss_res = np.sum((log_N[valid] - fitted) ** 2)
    ss_tot = np.sum((log_N[valid] - log_N[valid].mean()) ** 2)
    r2     = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return D_box, epsilons, Ns, r2


# ─────────────────────────────────────────────────────────────────────────────
# SVG  (original, unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def to_svg_coords(Z, x0, y0, w, h):
    xr, yr = Z.real, Z.imag
    x_lo, x_hi = np.percentile(xr, [1, 99])
    y_lo, y_hi = np.percentile(yr, [1, 99])
    span   = max(x_hi - x_lo, y_hi - y_lo, 1e-12)
    cx, cy = (x_lo + x_hi) / 2, (y_lo + y_hi) / 2
    scale  = min(w, h) / span * 0.98
    sx     = x0 + w / 2 + (xr - cx) * scale
    sy     = y0 + h / 2 - (yr - cy) * scale
    return sx, sy


def make_svg(ticker, Z, q, D_box, r2):
    x0, y0 = MARGIN, MARGIN
    w,  h  = W - 2 * MARGIN, H - 2 * MARGIN
    sx, sy = to_svg_coords(Z, x0, y0, w, h)
    pts    = " L".join(f"{x:.3f},{y:.3f}" for x, y in zip(sx, sy))
    dim_str = (f"D_box = {D_box:.4f}  (R² = {r2:.4f})"
               if np.isfinite(D_box) else "D_box = N/A")

    # Compute where the origin (0,0) maps to in SVG coordinates
    xr, yr = Z.real, Z.imag
    x_lo, x_hi = np.percentile(xr, [1, 99])
    y_lo, y_hi = np.percentile(yr, [1, 99])
    span   = max(x_hi - x_lo, y_hi - y_lo, 1e-12)
    cx, cy = (x_lo + x_hi) / 2, (y_lo + y_hi) / 2
    scale  = min(w, h) / span * 0.98

    # SVG pixel position of the data origin (0, 0)
    ox = x0 + w / 2 + (0 - cx) * scale
    oy = y0 + h / 2 - (0 - cy) * scale

    return f"""<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg"
     width="{W}" height="{H}" viewBox="0 0 {W} {H}">
  <rect width="{W}" height="{H}" fill="white"/>

  <!-- Real axis (horizontal through origin) -->
  <line x1="{x0}" y1="{oy:.2f}" x2="{x0+w}" y2="{oy:.2f}"
        stroke="#aaaaaa" stroke-width="1.5" stroke-dasharray="12,8"/>
  <!-- Imaginary axis (vertical through origin) -->
  <line x1="{ox:.2f}" y1="{y0}" x2="{ox:.2f}" y2="{y0+h}"
        stroke="#aaaaaa" stroke-width="1.5" stroke-dasharray="12,8"/>

  <!-- Axis labels -->
  <text x="{x0+w-10}" y="{oy:.2f}-12" text-anchor="end"
        font-size="22" fill="#888888">Re</text>
  <text x="{ox:.2f}+14" y="{y0+30}"
        font-size="22" fill="#888888">Im</text>

  <path d="M{pts}" stroke="black" stroke-width="0.25" fill="none"/>
  <text x="{W//2}" y="60" text-anchor="middle" font-size="28" fill="black">
    \u03a6(t; q={q}) \u2014 {ticker}
  </text>
  <text x="{W//2}" y="100" text-anchor="middle" font-size="24" fill="#555">
    {dim_str}
  </text>
</svg>"""


# ─────────────────────────────────────────────────────────────────────────────
# MATPLOTLIB PNG — crisp zoomed renders via data-space axis limits
# ─────────────────────────────────────────────────────────────────────────────
def save_zoomed_pngs_mpl(Z, ticker, q, D_box, r2, ticker_dir, base_name):
    xr, yr = Z.real, Z.imag

    x_lo, x_hi = np.percentile(xr, [1, 99])
    y_lo, y_hi = np.percentile(yr, [1, 99])
    cx = (x_lo + x_hi) / 2
    cy = (y_lo + y_hi) / 2
    half_full = max(x_hi - x_lo, y_hi - y_lo) / 2 * 1.02

    dim_str = (f"D_box = {D_box:.4f}   R² = {r2:.4f}"
               if np.isfinite(D_box) else "D_box = N/A")

    for label, frac in ZOOM_LEVELS:
        half = half_full * frac

        fig, ax = plt.subplots(figsize=(PNG_SIZE_IN, PNG_SIZE_IN), dpi=PNG_DPI)
        fig.patch.set_facecolor("white")

        ax.plot(xr, yr,
                color="black",
                linewidth=LINE_WIDTH,
                antialiased=True,
                rasterized=False)

        # ── Re/Im axes + 16x inset on full-extent only ──
        if label == "z1_full":
            ax.axhline(0, color="#aaaaaa", linewidth=1.0,
                       linestyle="--", dashes=(8, 5), zorder=0)
            ax.axvline(0, color="#aaaaaa", linewidth=1.0,
                       linestyle="--", dashes=(8, 5), zorder=0)
            ax.text(cx + half_full * 0.95, 0, r"Re$[\Phi]$",
                    va="bottom", ha="right", fontsize=24, color="#888888")
            ax.text(0, cy + half_full * 0.95, r"Im$[\Phi]$",
                    va="top", ha="left", fontsize=24, color="#888888")

            # Red rectangle on main plot showing where 16x zoom is
            half_16x = half_full * 0.0625
            from matplotlib.patches import Rectangle
            rect = Rectangle((cx - half_16x, cy - half_16x),
                              2 * half_16x, 2 * half_16x,
                              linewidth=1.2, edgecolor="#e63946",
                              facecolor="none", zorder=5)
            ax.add_patch(rect)

            # Inset in bottom-right corner — 16x zoom
            inset_ax = ax.inset_axes([0.67, 0.02, 0.30, 0.30])
            inset_ax.plot(xr, yr,
                          color="black",
                          linewidth=LINE_WIDTH * 0.6,
                          antialiased=True,
                          rasterized=False)
            inset_ax.set_xlim(cx - half_16x, cx + half_16x)
            inset_ax.set_ylim(cy - half_16x, cy + half_16x)
            inset_ax.set_aspect("equal", adjustable="box")
            inset_ax.set_facecolor("#f8f8f8")
            inset_ax.tick_params(left=False, bottom=False,
                                 labelleft=False, labelbottom=False)
            for spine in inset_ax.spines.values():
                spine.set_edgecolor("#e63946")
                spine.set_linewidth(1.2)
            inset_ax.set_title("16× zoom", fontsize=15, color="#e63946",
                   pad=3, fontweight="bold")

        ax.set_xlim(cx - half, cx + half)
        ax.set_ylim(cy - half, cy + half)
        ax.set_aspect("equal", adjustable="box")
        ax.axis("off")

        fig.tight_layout(pad=0.4)
        out_path = os.path.join(ticker_dir, f"{base_name}_{label}.png")
        fig.savefig(out_path, dpi=PNG_DPI, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        print(f"    saved {os.path.relpath(out_path)}")
# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

results = []

header = f"{'Ticker':<12} {'q':>5}  {'D_box':>8}  {'R²':>8}  {'N_pts':>8}"
print(header)
print("-" * len(header))

for ticker in TICKERS:
    print(f"\nProcessing {ticker} …")
    ret = get_returns(ticker)

    if len(ret) < 500:
        print(f"  [SKIP] only {len(ret)} return observations")
        continue

    safe_ticker = ticker.replace("-", "_").replace("=", "_")
    ticker_dir  = os.path.join(OUT_DIR, safe_ticker)
    os.makedirs(ticker_dir, exist_ok=True)

    for q in Q_VALUES:
        Z = phi(ret, q)
        D_box, epsilons, Ns, r2 = box_counting_dimension(Z)

        d_str = f"{D_box:8.4f}" if np.isfinite(D_box) else "     nan"
        r_str = f"{r2:8.4f}"   if np.isfinite(r2)    else "     nan"
        print(f"  {ticker:<10} {q:>5}  {d_str}  {r_str}  {len(Z):>8}")

        results.append({
            'Ticker': ticker,
            'q': q,
            'D_box': D_box,
            'R2': r2,
            'N_pts': len(Z)
        })

        base_name = f"{safe_ticker}_q{q}"

        # SVG
        svg_str  = make_svg(ticker, Z, q, D_box, r2)
        svg_path = os.path.join(ticker_dir, f"{base_name}.svg")
        with open(svg_path, "w", encoding="utf-8") as f:
            f.write(svg_str)

        # PNGs — matplotlib, data-space zoom
        save_zoomed_pngs_mpl(Z, ticker, q, D_box, r2, ticker_dir, base_name)

# Save the results to CSV
df = pd.DataFrame(results)
df.to_csv('fractal_dimensions.csv', index=False)

print("\nDone. Output in:", OUT_DIR)
print("Structure:")
print("  phi_full_zoom/")
print("    <TICKER>/")
print("      <TICKER>_q<Q>.svg")
print("      <TICKER>_q<Q>_z1_full.png    ← full extent")
print("      <TICKER>_q<Q>_z2_2x.png     ← 2× zoom")
print("      <TICKER>_q<Q>_z3_4x.png     ← 4× zoom")
print("      <TICKER>_q<Q>_z4_8x.png     ← 8× zoom")
print("      <TICKER>_q<Q>_z5_16x.png    ← 16× zoom")
print("CSV file saved: fractal_dimensions.csv")

Ticker           q     D_box        R²     N_pts
------------------------------------------------

Processing AAPL …
  AAPL           1    0.9962    1.0000      4091


C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\4176643946.py:75: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


    saved phi_full_zoom\AAPL\AAPL_q1_z1_full.png
    saved phi_full_zoom\AAPL\AAPL_q1_z2_2x.png
    saved phi_full_zoom\AAPL\AAPL_q1_z3_4x.png
    saved phi_full_zoom\AAPL\AAPL_q1_z4_8x.png
    saved phi_full_zoom\AAPL\AAPL_q1_z5_16x.png
  AAPL          10    1.0110    0.9999      4091
    saved phi_full_zoom\AAPL\AAPL_q10_z1_full.png
    saved phi_full_zoom\AAPL\AAPL_q10_z2_2x.png
    saved phi_full_zoom\AAPL\AAPL_q10_z3_4x.png
    saved phi_full_zoom\AAPL\AAPL_q10_z4_8x.png
    saved phi_full_zoom\AAPL\AAPL_q10_z5_16x.png
  AAPL          50    1.0579    0.9998      4091
    saved phi_full_zoom\AAPL\AAPL_q50_z1_full.png
    saved phi_full_zoom\AAPL\AAPL_q50_z2_2x.png
    saved phi_full_zoom\AAPL\AAPL_q50_z3_4x.png
    saved phi_full_zoom\AAPL\AAPL_q50_z4_8x.png
    saved phi_full_zoom\AAPL\AAPL_q50_z5_16x.png
  AAPL          75    1.0820    0.9996      4091
    saved phi_full_zoom\AAPL\AAPL_q75_z1_full.png
    saved phi_full_zoom\AAPL\AAPL_q75_z2_2x.png
    saved phi_full_zoom\AAPL\AA

C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\4176643946.py:75: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


    saved phi_full_zoom\MSFT\MSFT_q1_z1_full.png
    saved phi_full_zoom\MSFT\MSFT_q1_z2_2x.png
    saved phi_full_zoom\MSFT\MSFT_q1_z3_4x.png
    saved phi_full_zoom\MSFT\MSFT_q1_z4_8x.png
    saved phi_full_zoom\MSFT\MSFT_q1_z5_16x.png
  MSFT          10    1.0066    0.9999      4091
    saved phi_full_zoom\MSFT\MSFT_q10_z1_full.png
    saved phi_full_zoom\MSFT\MSFT_q10_z2_2x.png
    saved phi_full_zoom\MSFT\MSFT_q10_z3_4x.png
    saved phi_full_zoom\MSFT\MSFT_q10_z4_8x.png
    saved phi_full_zoom\MSFT\MSFT_q10_z5_16x.png
  MSFT          50    1.0486    0.9997      4091
    saved phi_full_zoom\MSFT\MSFT_q50_z1_full.png
    saved phi_full_zoom\MSFT\MSFT_q50_z2_2x.png
    saved phi_full_zoom\MSFT\MSFT_q50_z3_4x.png
    saved phi_full_zoom\MSFT\MSFT_q50_z4_8x.png
    saved phi_full_zoom\MSFT\MSFT_q50_z5_16x.png
  MSFT          75    1.0735    0.9997      4091
    saved phi_full_zoom\MSFT\MSFT_q75_z1_full.png
    saved phi_full_zoom\MSFT\MSFT_q75_z2_2x.png
    saved phi_full_zoom\MSFT\MS

...

    saved phi_full_zoom\FXI\FXI_q1_z1_full.png
    saved phi_full_zoom\FXI\FXI_q1_z2_2x.png
    saved phi_full_zoom\FXI\FXI_q1_z3_4x.png
    saved phi_full_zoom\FXI\FXI_q1_z4_8x.png
    saved phi_full_zoom\FXI\FXI_q1_z5_16x.png
  FXI           10    1.0061    0.9999      4091
    saved phi_full_zoom\FXI\FXI_q10_z1_full.png
    saved phi_full_zoom\FXI\FXI_q10_z2_2x.png
    saved phi_full_zoom\FXI\FXI_q10_z3_4x.png
    saved phi_full_zoom\FXI\FXI_q10_z4_8x.png
    saved phi_full_zoom\FXI\FXI_q10_z5_16x.png
  FXI           50    1.0593    0.9993      4091
    saved phi_full_zoom\FXI\FXI_q50_z1_full.png
    saved phi_full_zoom\FXI\FXI_q50_z2_2x.png
    saved phi_full_zoom\FXI\FXI_q50_z3_4x.png
    saved phi_full_zoom\FXI\FXI_q50_z4_8x.png
    saved phi_full_zoom\FXI\FXI_q50_z5_16x.png
  FXI           75    1.0919    0.9993      4091
    saved phi_full_zoom\FXI\FXI_q75_z1_full.png
    saved phi_full_zoom\FXI\FXI_q75_z2_2x.png
    saved phi_full_zoom\FXI\FXI_q75_z3_4x.png
    saved phi_full_

C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\4176643946.py:75: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


    saved phi_full_zoom\EWG\EWG_q1_z1_full.png
    saved phi_full_zoom\EWG\EWG_q1_z2_2x.png
    saved phi_full_zoom\EWG\EWG_q1_z3_4x.png
    saved phi_full_zoom\EWG\EWG_q1_z4_8x.png
    saved phi_full_zoom\EWG\EWG_q1_z5_16x.png
  EWG           10    1.0056    0.9999      4091
    saved phi_full_zoom\EWG\EWG_q10_z1_full.png
    saved phi_full_zoom\EWG\EWG_q10_z2_2x.png
    saved phi_full_zoom\EWG\EWG_q10_z3_4x.png
    saved phi_full_zoom\EWG\EWG_q10_z4_8x.png
    saved phi_full_zoom\EWG\EWG_q10_z5_16x.png
  EWG           50    1.0440    0.9998      4091
    saved phi_full_zoom\EWG\EWG_q50_z1_full.png
    saved phi_full_zoom\EWG\EWG_q50_z2_2x.png
    saved phi_full_zoom\EWG\EWG_q50_z3_4x.png
    saved phi_full_zoom\EWG\EWG_q50_z4_8x.png
    saved phi_full_zoom\EWG\EWG_q50_z5_16x.png
  EWG           75    1.0621    0.9994      4091
    saved phi_full_zoom\EWG\EWG_q75_z1_full.png
    saved phi_full_zoom\EWG\EWG_q75_z2_2x.png
    saved phi_full_zoom\EWG\EWG_q75_z3_4x.png
    saved phi_full_

C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\4176643946.py:75: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


    saved phi_full_zoom\EWU\EWU_q1_z1_full.png
    saved phi_full_zoom\EWU\EWU_q1_z2_2x.png
    saved phi_full_zoom\EWU\EWU_q1_z3_4x.png
    saved phi_full_zoom\EWU\EWU_q1_z4_8x.png
    saved phi_full_zoom\EWU\EWU_q1_z5_16x.png
  EWU           10    1.0035    0.9999      4091
    saved phi_full_zoom\EWU\EWU_q10_z1_full.png
    saved phi_full_zoom\EWU\EWU_q10_z2_2x.png
    saved phi_full_zoom\EWU\EWU_q10_z3_4x.png
    saved phi_full_zoom\EWU\EWU_q10_z4_8x.png
    saved phi_full_zoom\EWU\EWU_q10_z5_16x.png
  EWU           50    1.0375    0.9998      4091
    saved phi_full_zoom\EWU\EWU_q50_z1_full.png
    saved phi_full_zoom\EWU\EWU_q50_z2_2x.png
    saved phi_full_zoom\EWU\EWU_q50_z3_4x.png
    saved phi_full_zoom\EWU\EWU_q50_z4_8x.png
    saved phi_full_zoom\EWU\EWU_q50_z5_16x.png
  EWU           75    1.0565    0.9997      4091
    saved phi_full_zoom\EWU\EWU_q75_z1_full.png
    saved phi_full_zoom\EWU\EWU_q75_z2_2x.png
    saved phi_full_zoom\EWU\EWU_q75_z3_4x.png
    saved phi_full_

In [1]:
"""
Φ(t; q) — Full Canvas SVG + Box-Counting Fractal Dimension
=============================================================
- Generates SVG path for each ticker × q combination
- Computes D_box (box-counting / block partition fractal dimension)
- Prints dimension table to stdout
- White background, black line, robust percentile scaling
"""

import numpy as np
import yfinance as yf
import os

TICKERS = [
    # US Large Cap Tech
    "AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "AMD", "INTC",

    # US Financials
    "JPM", "BAC", "GS", "MS", "WFC",
    "BLK", "C", "AXP", "V", "MA",

    # US Healthcare & Consumer
    "JNJ", "PFE", "UNH", "MRK", "ABBV",
    "PG", "KO", "PEP", "WMT", "COST",

    # US Energy & Industrials
    "XOM", "CVX", "COP", "BA", "CAT",
    "GE", "HON", "LMT", "RTX", "UPS",

    # Commodities & Crypto (yfinance format)
    "GC=F",    # Gold
    "SI=F",    # Silver
    "CL=F",    # Crude Oil
    "BTC-USD", # Bitcoin
    "ETH-USD", # Ethereum

    # International ETFs (broad exposure)
    "EWJ",  # Japan
    "EWZ",  # Brazil
    "FXI",  # China
    "EWG",  # Germany
    "EWU",  # UK
]
START   = "2010-01-01"
END     = "2026-04-11"

Q_VALUES = [1, 10, 50, 75, 100, 150, 200]

OUT_DIR = "phi_full_zoom"
os.makedirs(OUT_DIR, exist_ok=True)

W, H   = 8000, 8000
MARGIN = 100


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────

def get_returns(ticker):
    df  = yf.download(ticker, start=START, end=END, progress=False)
    px  = df["Close"].dropna().values.ravel().astype(float)
    px  = px[px > 0]
    ret = np.diff(np.log(px))
    return ret[np.isfinite(ret)]


# ─────────────────────────────────────────────────────────────────────────────
# Φ CURVE
# ─────────────────────────────────────────────────────────────────────────────

def phi(ret, q):
    """Cumulative characteristic function path in the complex plane."""
    return np.cumsum(np.exp(1j * q * ret))


# ─────────────────────────────────────────────────────────────────────────────
# BOX-COUNTING FRACTAL DIMENSION
# ─────────────────────────────────────────────────────────────────────────────

def box_counting_dimension(Z, n_scales=20, scale_min=None, scale_max=None):
    """
    Block-partition / box-counting fractal dimension of complex path Z.

    Normalises Z into [0,1]² then, for each box side ε, counts the number
    of distinct grid cells N(ε) visited by the curve.  The OLS slope of
    log N vs log(1/ε) is D_box.

    Parameters
    ----------
    Z         : complex ndarray
    n_scales  : number of ε values to probe
    scale_min : smallest box side (fraction of span); default ≈ 2 inter-point gaps
    scale_max : largest  box side (fraction of span); default 0.20

    Returns
    -------
    D_box    : fractal dimension (nan if fit fails)
    epsilons : box sizes used
    Ns       : N(ε) counts
    r2       : R² of the log-log fit
    """
    xr, yr = Z.real, Z.imag

    x_span = xr.max() - xr.min()
    y_span = yr.max() - yr.min()
    span   = max(x_span, y_span, 1e-12)

    # Normalise to [0, 1]²
    xn = (xr - xr.min()) / span
    yn = (yr - yr.min()) / span

    if scale_max is None:
        scale_max = 0.20
    if scale_min is None:
        scale_min = 2.0 / len(Z)

    epsilons = np.logspace(
        np.log10(scale_min), np.log10(scale_max), n_scales
    )

    Ns = []
    for eps in epsilons:
        ix       = np.floor(xn / eps).astype(np.int64)
        iy       = np.floor(yn / eps).astype(np.int64)
        occupied = len(set(zip(ix.tolist(), iy.tolist())))
        Ns.append(occupied)

    Ns      = np.array(Ns, dtype=float)
    log_eps = np.log(epsilons)
    log_N   = np.log(Ns)

    valid = np.isfinite(log_N) & (Ns > 1)
    if valid.sum() < 4:
        return np.nan, epsilons, Ns, np.nan

    coeffs = np.polyfit(log_eps[valid], log_N[valid], 1)
    D_box  = -coeffs[0]          # N(ε) ~ ε^{-D}  →  slope of log N vs log ε = −D

    fitted = np.polyval(coeffs, log_eps[valid])
    ss_res = np.sum((log_N[valid] - fitted) ** 2)
    ss_tot = np.sum((log_N[valid] - log_N[valid].mean()) ** 2)
    r2     = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return D_box, epsilons, Ns, r2


# ─────────────────────────────────────────────────────────────────────────────
# SVG HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def to_svg_coords(Z, x0, y0, w, h):
    xr, yr = Z.real, Z.imag

    x_low,  x_high = np.percentile(xr, [1, 99])
    y_low,  y_high = np.percentile(yr, [1, 99])

    span = max(x_high - x_low, y_high - y_low, 1e-12)
    cx   = (x_low + x_high) / 2
    cy   = (y_low + y_high) / 2

    scale = min(w, h) / span * 0.98

    sx = x0 + w / 2 + (xr - cx) * scale
    sy = y0 + h / 2 - (yr - cy) * scale   # flip y for SVG

    return sx, sy


def make_svg(ticker, Z, q, D_box, r2):
    x0, y0 = MARGIN, MARGIN
    w,  h  = W - 2 * MARGIN, H - 2 * MARGIN

    sx, sy = to_svg_coords(Z, x0, y0, w, h)

    pts = " L".join(f"{x:.3f},{y:.3f}" for x, y in zip(sx, sy))

    dim_str = f"D_box = {D_box:.4f}  (R² = {r2:.4f})" if np.isfinite(D_box) else "D_box = N/A"

    svg = f"""<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg"
     width="{W}" height="{H}"
     viewBox="0 0 {W} {H}">

  <rect width="{W}" height="{H}" fill="white"/>

  <path d="M{pts}"
        stroke="black"
        stroke-width="0.25"
        fill="none"/>

  <text x="{W // 2}" y="60" text-anchor="middle"
        font-size="28" fill="black">
    \u03a6(t; q={q}) \u2014 {ticker}
  </text>

  <text x="{W // 2}" y="100" text-anchor="middle"
        font-size="24" fill="#555">
    {dim_str}
  </text>

</svg>"""

    return svg


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

header = f"{'Ticker':<12} {'q':>5}  {'D_box':>8}  {'R²':>8}  {'N_pts':>8}"
print(header)
print("-" * len(header))

for ticker in TICKERS:
    print(f"\nProcessing {ticker} ...")
    ret = get_returns(ticker)

    if len(ret) < 500:
        print(f"  [SKIP] only {len(ret)} return observations")
        continue

    for q in Q_VALUES:
        Z = phi(ret, q)

        D_box, epsilons, Ns, r2 = box_counting_dimension(Z)

        # ── print to stdout ──────────────────────────────────────────────
        d_str = f"{D_box:8.4f}" if np.isfinite(D_box) else "     nan"
        r_str = f"{r2:8.4f}"   if np.isfinite(r2)    else "     nan"
        print(f"  {ticker:<10} {q:>5}  {d_str}  {r_str}  {len(Z):>8}")

        # ── save SVG ─────────────────────────────────────────────────────
        svg   = make_svg(ticker, Z, q, D_box, r2)
        fname = os.path.join(
            OUT_DIR,
            f"{ticker.replace('-', '_')}_q{q}.svg"
        )
        with open(fname, "w", encoding="utf-8") as f:
            f.write(svg)

print("\nDone. SVGs saved to:", OUT_DIR)

Ticker           q     D_box        R²     N_pts
------------------------------------------------

Processing AAPL ...


C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\1125115528.py:62: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


  AAPL           1    0.9962    1.0000      4091
  AAPL          10    1.0110    0.9999      4091
  AAPL          50    1.0579    0.9998      4091
  AAPL          75    1.0820    0.9996      4091
  AAPL         100    1.1043    0.9993      4091
  AAPL         150    1.1363    0.9957      4091
  AAPL         200    1.1830    0.9840      4091

Processing MSFT ...


C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\1125115528.py:62: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


  MSFT           1    0.9964    1.0000      4091
  MSFT          10    1.0066    0.9999      4091
  MSFT          50    1.0486    0.9997      4091
  MSFT          75    1.0735    0.9997      4091
  MSFT         100    1.0939    0.9996      4091
  MSFT         150    1.1309    0.9971      4091
  MSFT         200    1.1718    0.9881      4091

Processing GOOG ...


...

  FXI            1    0.9958    1.0000      4091
  FXI           10    1.0061    0.9999      4091
  FXI           50    1.0593    0.9993      4091
  FXI           75    1.0919    0.9993      4091
  FXI          100    1.1197    0.9992      4091
  FXI          150    1.1652    0.9932      4091
  FXI          200    1.2010    0.9721      4091

Processing EWG ...


C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\1125115528.py:62: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


  EWG            1    0.9958    1.0000      4091
  EWG           10    1.0056    0.9999      4091
  EWG           50    1.0440    0.9998      4091
  EWG           75    1.0621    0.9994      4091
  EWG          100    1.0818    0.9997      4091
  EWG          150    1.1058    0.9984      4091
  EWG          200    1.1370    0.9941      4091

Processing EWU ...


C:\Users\VEDANT\AppData\Local\Temp\ipykernel_9120\1125115528.py:62: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df  = yf.download(ticker, start=START, end=END, progress=False)


  EWU            1    0.9958    1.0000      4091
  EWU           10    1.0035    0.9999      4091
  EWU           50    1.0375    0.9998      4091
  EWU           75    1.0565    0.9997      4091
  EWU          100    1.0675    0.9997      4091
  EWU          150    1.0913    0.9992      4091
  EWU          200    1.1129    0.9978      4091

Done. SVGs saved to: phi_full_zoom


In [ ]:
"""
Fractal Structure Analysis of Financial Time Series
=====================================================

Three analysis modules per ticker:

  1. CUMULATIVE CHARACTERISTIC ANALYSIS (first order)
     - Characteristic curves F_q from returns
     
  2. MF-DFA (standard multifractal analysis)
  
  3. SECOND-ORDER CHARACTERISTIC ANALYSIS (NEW)
     - Treat each F_q curve as a complex time series
     - Compute ITS characteristic curves G_{q2}(F_q)
     - Analyze fractal structure of the characteristic curves themselves
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import savgol_filter
from scipy import stats
import os
import warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
TICKERS   = ["AAPL", "MSFT", "GOOG", "AMZN", "NVDA", "JPM", "XOM", "BTC-USD"]
START     = "2010-01-01"
END       = "2026-03-01"
OUT_DIR   = "fractal_structure_results"

# q values for cumulative characteristic (wavenumber analogue)
Q_CHAR         = np.array([0, 1, 5, 10, 20, 60, 120, 200, 300], dtype=float)

# Second-order q values (for analyzing the characteristic curves themselves)
Q_SECOND_ORDER = np.array([0, 1, 2, 5, 10, 20], dtype=float)

# q values for MF-DFA
Q_HURST = np.arange(-30, 31, 5, dtype=float)

# MF-DFA scale parameters
SCALE_MIN = 20
SCALE_MAX = 800
N_SCALES  = 30
POLY_ORD  = 1
OVERLAP   = 0.75

# Palette for F_q(s) lines
CMAP_LINES = [
    "#c0392b", "#2980b9", "#27ae60", "#8e44ad",
    "#d35400", "#16a085", "#2c3e50", "#e67e22",
    "#1abc9c", "#e74c3c", "#3498db",
]

os.makedirs(OUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# STYLE
# ─────────────────────────────────────────────────────────────────────────────
PLT_STYLE = dict(
    fig_fc    = "#ffffff",
    ax_fc     = "#ffffff",
    text      = "#1a1a1a",
    grid      = "#e5e5e5",
    spine     = "#cccccc",
    accent    = "#1a1a2e",
    line_main = "#1f4e8c",
    line_alt  = "#c0392b",
    line_grn  = "#1a7a4a",
)

def _style(fig, axes_flat):
    fig.patch.set_facecolor(PLT_STYLE["fig_fc"])
    for ax in axes_flat:
        ax.set_facecolor(PLT_STYLE["ax_fc"])
        ax.tick_params(colors=PLT_STYLE["text"], which="both",
                       direction="in", top=True, right=True, labelsize=8)
        ax.xaxis.label.set_color(PLT_STYLE["text"])
        ax.yaxis.label.set_color(PLT_STYLE["text"])
        ax.title.set_color(PLT_STYLE["accent"])
        for sp in ax.spines.values():
            sp.set_edgecolor(PLT_STYLE["spine"])
        ax.grid(True, color=PLT_STYLE["grid"], linewidth=0.5,
                linestyle="--", alpha=0.9)

def _legend(ax, **kw):
    return ax.legend(framealpha=0.9, facecolor="white",
                     edgecolor=PLT_STYLE["spine"],
                     labelcolor=PLT_STYLE["text"], **kw)


# ─────────────────────────────────────────────────────────────────────────────
# DIRECTORIES
# ─────────────────────────────────────────────────────────────────────────────
def get_dirs(ticker):
    t    = ticker.replace("=", "_")
    base = os.path.join(OUT_DIR, t)
    char = os.path.join(base, "characteristic_curves")
    mfd  = os.path.join(base, "mfdfa")
    second = os.path.join(base, "second_order")
    os.makedirs(char, exist_ok=True)
    os.makedirs(mfd,  exist_ok=True)
    os.makedirs(second, exist_ok=True)
    return char, mfd, second


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────
def get_data(ticker):
    df = yf.download(ticker, start=START, end=END, progress=False)
    if df is None or len(df) == 0:
        return None, None
    try:
        if hasattr(df.columns, "levels"):
            px = df["Close"][ticker].dropna().values
        else:
            px = df["Close"].dropna().values
    except Exception:
        px = df["Close"].dropna().values

    px = px.ravel().astype(float)
    px = px[px > 0]
    if len(px) < 200:
        return None, None

    logp = np.log(px)
    ret  = np.diff(logp)
    ret  = ret[np.isfinite(ret)]
    if len(ret) < 200:
        return None, None
    return logp, ret


# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA
# ─────────────────────────────────────────────────────────────────────────────
def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


def mfdfa(x, q_vals, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
          n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP):
    """MF-DFA on log-returns with overlapping segments."""
    x  = x[np.isfinite(x)]
    Y  = np.cumsum(x - x.mean())
    N  = len(Y)

    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue
        t   = np.arange(s)
        rms = []
        for st in starts:
            seg  = Y[st: st + s]
            coef = np.polyfit(t, seg, poly_ord)
            r_v  = np.sqrt(np.mean((seg - np.polyval(coef, t)) ** 2))
            if r_v > 0:
                rms.append(r_v)
        rms = np.asarray(rms)
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms ** 2)))
            else:
                Fqs[j, i] = (np.mean(rms ** q)) ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


def multifractal_spectrum(q_vals, hq):
    """τ(q), α(q), f(α) from H(q)."""
    tau_q = q_vals * hq - 1.0
    wl    = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_s = (savgol_filter(tau_q, window_length=wl, polyorder=2)
             if len(tau_q) >= 5 else tau_q)
    alpha   = np.gradient(tau_s, q_vals)
    D_alpha = q_vals * alpha - tau_s
    return tau_q, alpha, D_alpha


def hausdorff_from_spectrum(D_alpha):
    """True Hausdorff dimension from multifractal spectrum."""
    valid = np.isfinite(D_alpha)
    if valid.sum() < 2:
        return 1.0
    return float(np.clip(np.nanmax(D_alpha[valid]), 1.0, 2.0))


def hausdorff_boxcount(F, n_eps=15):
    """Box-counting dimension of a complex curve."""
    x, y = F.real, F.imag
    if len(x) < 10:
        return 1.0
    eps    = np.logspace(-1.0, -2.5, n_eps)
    counts = [len(set(zip((x / e).astype(int), (y / e).astype(int))))
              for e in eps]
    counts = np.array(counts, dtype=float)
    valid  = counts > 0
    if valid.sum() < 3:
        return 1.0
    slope, _ = np.polyfit(np.log(eps[valid]), np.log(counts[valid]), 1)
    return float(np.clip(-slope, 1.0, 2.0))


def lyapunov_mf(ret, q_vals, hq, alpha, D_alpha):
    """Estimate Lyapunov exponent from multifractal spectrum."""
    sigma = float(np.std(ret))

    q2_idx = np.argmin(np.abs(q_vals - 2.0))
    H2     = float(hq[q2_idx]) if np.isfinite(hq[q2_idx]) else 0.5

    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    if valid_s.sum() >= 2:
        delta_alpha = float(alpha[valid_s].max() - alpha[valid_s].min())
        alpha0      = float(alpha[np.argmin(np.abs(q_vals))])
    else:
        delta_alpha = 0.0
        alpha0      = H2

    lam_mf = H2 * sigma * (1.0 + delta_alpha)
    lam_mf = float(np.clip(lam_mf, 1e-6, None))

    D_H_spectral = hausdorff_from_spectrum(D_alpha)

    return dict(
        lam_mf       = lam_mf,
        H2           = H2,
        sigma        = sigma,
        delta_alpha  = delta_alpha,
        alpha0       = alpha0,
        D_H_spectral = D_H_spectral,
    )


# ─────────────────────────────────────────────────────────────────────────────
# FIRST-ORDER CHARACTERISTIC CURVE
# ─────────────────────────────────────────────────────────────────────────────
def cumulative_characteristic(r, q, n_grid=2000):
    """Compute F_q = sum(e^{i*q*r}) for returns r."""
    r = np.asarray(r, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 10:
        return np.linspace(0, 1, n_grid) + 0j

    if len(r) > n_grid:
        idx = np.linspace(0, len(r) - 1, n_grid).astype(int)
        r_used = r[idx]
    else:
        r_used = r

    F = np.cumsum(np.exp(1j * q * r_used))

    d = F[-1] - F[0]
    if np.abs(d) < 1e-12:
        return F / (np.max(np.abs(F)) + 1e-12)
    return (F - F[0]) / d


# ─────────────────────────────────────────────────────────────────────────────
# SECOND-ORDER CHARACTERISTIC ANALYSIS
# Apply the same characteristic transform to the characteristic curves themselves
# ─────────────────────────────────────────────────────────────────────────────
def second_order_characteristic(curve, q2):
    """
    Compute characteristic curve of a characteristic curve.
    Treats the complex curve F_q as a complex-valued time series.
    """
    curve = np.asarray(curve)
    if len(curve) < 10:
        return np.array([0+0j])
    
    # curve is already complex (1D array of complex numbers)
    # Compute incremental steps as complex differences
    steps = np.diff(curve)  # This gives complex numbers directly
    
    # Apply characteristic transform to the steps of the curve
    G = np.cumsum(np.exp(1j * q2 * steps))
    
    # Normalize
    if len(G) > 1:
        d = G[-1] - G[0]
        if np.abs(d) < 1e-12:
            G = G / (np.max(np.abs(G)) + 1e-12)
        else:
            G = (G - G[0]) / d
    
    return G
def analyze_second_order(ticker, ret, char_dir, second_dir):
    """
    Analyze the fractal structure of the characteristic curves themselves.
    For each first-order F_q, compute its second-order characteristic curves G_{q2}.
    """
    print("\n  → SECOND-ORDER CHARACTERISTIC ANALYSIS")
    print("    (applying the same transform to the characteristic curves themselves)")
    
    # First, compute all first-order characteristic curves
    first_order_curves = {}
    print("    Computing first-order characteristic curves...")
    for q in Q_CHAR:
        F = cumulative_characteristic(ret, q)
        first_order_curves[q] = F
        print(f"      q={int(q)}: curve with {len(F)} points")
    
    # For each first-order q, compute second-order curves for different q2
    second_order_results = {}
    
    # Select a subset of first-order q for clarity in plots
    q_subset = [0, 5, 20, 60, 200]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    '''fig.suptitle(
        f"Second-Order Characteristic Analysis  ·  {ticker}\n"
        r"Characteristic curves of the characteristic curves $G_{q2}(F_q)$",
        fontsize=12, fontweight="bold", color=PLT_STYLE["accent"], y=1.02,
    )'''
    _style(fig, axes.ravel())
    
    # Panel (a): First-order curves (reference)
    ax = axes[0, 0]
    colors_first = plt.cm.viridis(np.linspace(0, 0.9, len(q_subset)))
    for idx, q in enumerate(q_subset):
        F = first_order_curves[q]
        ax.plot(F.real, F.imag, '-', color=colors_first[idx], 
               linewidth=1.5, alpha=0.8, label=f'F_{{{int(q)}}}')
    ax.set_xlabel('Re $F_q$', fontsize=10)
    ax.set_ylabel('Im $F_q$', fontsize=10)
    ax.set_title('(a) First-order characteristic curves $F_q$', fontsize=10)
    ax.set_aspect('equal')
    _legend(ax, fontsize=7, ncol=2)
    
    # Panel (b): Second-order curves for a fixed q (showing how G_{q2} varies with q2)
    ax = axes[0, 1]
    fixed_q = 20  # Pick a representative first-order q
    F_fixed = first_order_curves[fixed_q]
    
    colors_second = plt.cm.plasma(np.linspace(0, 0.9, len(Q_SECOND_ORDER)))
    for idx, q2 in enumerate(Q_SECOND_ORDER):
        G = second_order_characteristic(F_fixed, q2)
        ax.plot(G.real, G.imag, '-', color=colors_second[idx], 
               linewidth=1.5, alpha=0.8, label=f'$G_{{{int(q2)}}}(F_{{{fixed_q}}})$')
        # Compute fractal dimension of this second-order curve
        D_second = hausdorff_boxcount(G)
        print(f"      q={fixed_q}, q2={int(q2)}: D_H(G) = {D_second:.4f}")
    
    ax.set_xlabel('Re $G_{q2}$', fontsize=10)
    ax.set_ylabel('Im $G_{q2}$', fontsize=10)
    ax.set_title(f'(b) Second-order curves $G_{{q2}}(F_{{{fixed_q}}})$', fontsize=10)
    ax.set_aspect('equal')
    _legend(ax, fontsize=6, ncol=1, loc='center right')
    
    # Panel (c): Fractal dimension of second-order curves vs q2 for multiple first-order q
    ax = axes[1, 0]
    
    for idx, q in enumerate(q_subset):
        F = first_order_curves[q]
        D_vals = []
        for q2 in Q_SECOND_ORDER:
            G = second_order_characteristic(F, q2)
            D = hausdorff_boxcount(G)
            D_vals.append(D)
        ax.plot(Q_SECOND_ORDER, D_vals, 'o-', color=colors_first[idx],
               linewidth=1.5, markersize=5, label=f'F_{{{int(q)}}}')
    
    ax.set_xlabel('Second-order wavenumber $q_2$', fontsize=10)
    ax.set_ylabel('Fractal dimension $D_H(G_{q2})$', fontsize=10)
    ax.set_title('(c) Dimension of second-order curves vs $q_2$', fontsize=10)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='D=1 (line)')
    ax.axhline(2.0, color='gray', linestyle='--', alpha=0.5, label='D=2 (area)')
    _legend(ax, fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)
    
    # Panel (d): D_H(G) vs D_H(F) - relationship between first and second order dimensions
    ax = axes[1, 1]
    
    # For a fixed q2, plot second-order dimension vs first-order dimension
    fixed_q2 = 5
    first_dims = []
    second_dims = []
    
    for q in q_subset:
        F = first_order_curves[q]
        D_first = hausdorff_boxcount(F)
        G = second_order_characteristic(F, fixed_q2)
        D_second = hausdorff_boxcount(G)
        first_dims.append(D_first)
        second_dims.append(D_second)
        ax.annotate(f'q={int(q)}', xy=(D_first, D_second), xytext=(3, 3),
                   textcoords='offset points', fontsize=8)
    
    ax.scatter(first_dims, second_dims, s=80, color=PLT_STYLE["line_main"],
              edgecolors='black', linewidth=1, alpha=0.7, zorder=5)
    
    # Fit line
    if len(first_dims) >= 3:
        slope, intercept, r_value, _, _ = stats.linregress(first_dims, second_dims)
        x_line = np.linspace(min(first_dims), max(first_dims), 100)
        ax.plot(x_line, intercept + slope * x_line, '--',
               color=PLT_STYLE["line_alt"], linewidth=1.5,
               label=f'D_H(G) = {intercept:.3f} + {slope:.3f}·D_H(F)\nR²={r_value**2:.3f}')
    
    ax.set_xlabel('Fractal dimension of $F_q$', fontsize=10)
    ax.set_ylabel(f'Fractal dimension of $G_{{{fixed_q2}}}(F_q)$', fontsize=10)
    ax.set_title(f'(d) First-order vs second-order dimension ($q_2={fixed_q2}$)', fontsize=10)
    _legend(ax, fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    fname = os.path.join(second_dir, f"{ticker.replace('=','_')}_second_order.png")
    plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")
    
    # Also create individual plots for each first-order q
    for q in q_subset:
        fig, ax = plt.subplots(figsize=(10, 8))
        F = first_order_curves[q]
        
        colors_second = plt.cm.plasma(np.linspace(0, 0.9, len(Q_SECOND_ORDER)))
        for idx, q2 in enumerate(Q_SECOND_ORDER):
            G = second_order_characteristic(F, q2)
            D = hausdorff_boxcount(G)
            ax.plot(G.real, G.imag, '-', color=colors_second[idx],
                   linewidth=1.5, alpha=0.8, 
                   label=f'$G_{{{int(q2)}}}$, D={D:.3f}')
        
        ax.set_xlabel('Re $G_{q2}$', fontsize=12)
        ax.set_ylabel('Im $G_{q2}$', fontsize=12)
        ax.set_title(f'{ticker}: Second-order curves $G_{{q2}}(F_{{{int(q)}}})$', 
                    fontsize=14, fontweight='bold')
        ax.set_aspect('equal')
        _legend(ax, fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        fname = os.path.join(second_dir, f"{ticker.replace('=','_')}_second_order_q{q}.png")
        plt.savefig(fname, dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig)
    
    return second_order_results

# ─────────────────────────────────────────────────────────────────────────────
# PLOT A — FIRST-ORDER CUMULATIVE CHARACTERISTIC
# ─────────────────────────────────────────────────────────────────────────────
def plot_characteristic(ticker, ret, lam_dict, char_dir):
    """4-panel figure for first-order cumulative characteristic analysis."""
    DH_geom = np.array([hausdorff_boxcount(cumulative_characteristic(ret, q))
                        for q in Q_CHAR])
    q2      = Q_CHAR ** 2
    valid_m = np.isfinite(DH_geom) & (q2 > 0)

    slope_fit, intercept_fit = (np.polyfit(q2[valid_m], DH_geom[valid_m], 1)
                                if valid_m.sum() >= 2 else (np.nan, 1.0))

    lam_mf       = lam_dict["lam_mf"]
    H2           = lam_dict["H2"]
    sigma        = lam_dict["sigma"]
    delta_alpha  = lam_dict["delta_alpha"]
    D_H_spectral = lam_dict["D_H_spectral"]

    D_implied = slope_fit * lam_mf if np.isfinite(slope_fit) else np.nan

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"First-Order Characteristic Analysis  ·  {ticker}",
        fontsize=13, fontweight="bold", color=PLT_STYLE["accent"], y=1.01,
    )
    _style(fig, axes.ravel())

    # (a) curves in ℂ
    ax = axes[0, 0]
    for i, q in enumerate(Q_CHAR):
        c = plt.cm.magma_r(i / len(Q_CHAR))
        F  = cumulative_characteristic(ret, q)
        lw = 1.0 if q > 20 else 1.6
        ax.plot(F.real, F.imag, linewidth=lw, color=c,
                label=f"$q={int(q)}$", alpha=0.88)
    ax.set_xlabel("Re $F_q$", fontsize=10)
    ax.set_ylabel("Im $F_q$", fontsize=10)
    ax.set_title("(a) First-order characteristic curves $F_q$", fontsize=10)
    _legend(ax, fontsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    ax.text(0.03, 0.97,
            fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="left", va="top", fontsize=8,
            color=PLT_STYLE["line_grn"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # (b) D_H_geom(q) vs q²
    ax = axes[0, 1]
    ax.scatter(q2, DH_geom, color=PLT_STYLE["line_main"], s=60, zorder=5,
               edgecolors="#aaaaaa", linewidths=0.5,
               label=r"$D_H^{\rm geom}(q)$  (box-count of $F_q$)")
    if np.isfinite(slope_fit):
        q2_line = np.linspace(0, q2.max() * 1.06, 200)
        ax.plot(q2_line, intercept_fit + slope_fit * q2_line,
                color=PLT_STYLE["line_alt"], linewidth=1.5, linestyle="--",
                label=fr"fit  slope$={slope_fit:.2e}$")
        if np.isfinite(D_implied):
            ax.plot(q2_line, 1.0 + (D_implied / lam_mf) * q2_line,
                    color=PLT_STYLE["line_grn"], linewidth=1.2, linestyle=":",
                    label=fr"$\hat{{D}}/\lambda_{{MF}}={D_implied/lam_mf:.2e}$")
    ax.axhline(D_H_spectral, color=PLT_STYLE["line_grn"], linewidth=1.2,
               linestyle="-.",
               label=fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$")
    ax.axhline(1.0, color="#aaaaaa", linewidth=0.8, linestyle=":")
    ax.set_xlabel("$q^2$",    fontsize=10)
    ax.set_ylabel(r"$D_H^{\rm geom}(q)$", fontsize=10)
    ax.set_title(r"(b) $D_H^{\rm geom}(q)$ vs $q^2$", fontsize=9)
    _legend(ax, fontsize=7)
    ax.text(0.97, 0.12,
            fr"$\lambda_{{MF}} = {lam_mf:.4f}$",
            transform=ax.transAxes, ha="right", fontsize=9,
            color=PLT_STYLE["line_main"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # (c) λ_MF component breakdown
    ax = axes[1, 0]
    labels  = ["$H(q{=}2)$", r"$\sigma$", r"$1+\Delta\alpha$", r"$\lambda_{MF}$"]
    values  = [H2, sigma, 1.0 + delta_alpha, lam_mf]
    colours = [PLT_STYLE["line_main"], PLT_STYLE["line_grn"],
               "#e67e22", PLT_STYLE["line_alt"]]
    bars = ax.bar(labels, values, color=colours, edgecolor="#cccccc",
                  linewidth=0.8, alpha=0.85, width=0.55)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.02,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=8, color=PLT_STYLE["text"])
    ax.set_ylabel("Value", fontsize=10)
    ax.set_title(
        r"(c)  $\lambda_{MF} = H(q{=}2)\cdot\sigma\cdot(1+\Delta\alpha)$",
        fontsize=10)
    ax.set_ylim(0, max(values) * 1.18)
    ax.text(0.97, 0.97,
            fr"$\lambda_{{MF}} = {H2:.3f}\times{sigma:.4f}\times{1+delta_alpha:.3f}$"
            f"\n$= {lam_mf:.5f}$"
            f"\n$D_H = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            color=PLT_STYLE["text"],
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    # (d) Implied diffusion coefficient
    ax = axes[1, 1]
    dh_excess = DH_geom - 1.0
    q2_scaled = q2 / lam_mf
    ax.scatter(q2_scaled[valid_m], dh_excess[valid_m],
               color=PLT_STYLE["line_main"], s=60, zorder=5,
               edgecolors="#aaaaaa", linewidths=0.5,
               label=r"$(D_H^{\rm geom} - 1)$ vs $q^2/\lambda_{MF}$")
    if np.isfinite(D_implied) and valid_m.sum() >= 2:
        x_line = np.linspace(0, q2_scaled[valid_m].max() * 1.06, 200)
        ax.plot(x_line, D_implied * x_line,
                color=PLT_STYLE["line_alt"], linewidth=1.5, linestyle="--",
                label=fr"slope $= \hat{{D}} = {D_implied:.4f}$")
    ax.axhline(0, color="#aaaaaa", linewidth=0.8, linestyle=":")
    ax.axhline(D_H_spectral - 1.0, color=PLT_STYLE["line_grn"],
               linewidth=1.0, linestyle="-.",
               label=fr"$D_H - 1 = {D_H_spectral-1:.4f}$  (spectral)")
    ax.set_xlabel(r"$q^2 / \lambda_{MF}$",  fontsize=10)
    ax.set_ylabel(r"$D_H^{\rm geom} - 1$",  fontsize=10)
    ax.set_title(r"(d) Implied diffusion coefficient $\hat{\mathcal{D}}$",
                 fontsize=10)
    _legend(ax, fontsize=7)
    if np.isfinite(D_implied):
        ax.text(0.97, 0.12,
                fr"$\hat{{\mathcal{{D}}}} = {D_implied:.5f}$",
                transform=ax.transAxes, ha="right", fontsize=9,
                color=PLT_STYLE["line_alt"],
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor=PLT_STYLE["spine"], alpha=0.85))

    plt.tight_layout()
    fname = os.path.join(char_dir, f"{ticker.replace('=','_')}_characteristic.png")
    plt.savefig(fname, dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT B — MF-DFA
# ─────────────────────────────────────────────────────────────────────────────
def plot_mfdfa(ticker, ret, scales, Fqs, hq, q_vals,
               tau_q, alpha, D_alpha, lam_dict, mfd_dir):
    valid   = np.isfinite(hq)
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    colors  = (CMAP_LINES * 5)[: len(q_vals)]

    lam_mf       = lam_dict["lam_mf"]
    H2           = lam_dict["H2"]
    delta_alpha  = lam_dict["delta_alpha"]
    alpha0       = lam_dict["alpha0"]
    D_H_spectral = lam_dict["D_H_spectral"]

    fig, axs = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(
        f"Multifractal DFA  ·  {ticker}  ·  log-returns",
        fontsize=13, fontweight="bold", color=PLT_STYLE["accent"], y=1.01,
    )
    _style(fig, axs.ravel())

    ax = axs[0, 0]
    ax.plot(ret, color=PLT_STYLE["line_main"], linewidth=0.55, alpha=0.9)
    ax.set_xlabel("$t$  (trading days)", fontsize=10)
    ax.set_ylabel("Log-return",           fontsize=10)
    ax.set_title("(a) Log-return series", fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3g"))

    ax = axs[0, 1]
    legend_handles = []
    for idx_q, (q, c) in enumerate(zip(q_vals, colors)):
        row  = Fqs[idx_q]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        h, = ax.loglog(scales[mask], row[mask], "o-",
                       color=c, markersize=2.8, linewidth=0.85, alpha=0.8)
        legend_handles.append((h, f"$q={int(q)}$"))
    ax.set_xlabel("Scale $s$", fontsize=10)
    ax.set_ylabel("$F_q(s)$",  fontsize=10)
    ax.set_title("(b) Fluctuation functions $F_q(s)$", fontsize=10)
    step  = max(1, len(legend_handles) // 7)
    shown = legend_handles[::step]
    if shown:
        _legend(ax, handles=[h for h, _ in shown],
                labels=[l for _, l in shown],
                fontsize=7, ncol=1, loc="best",
                handlelength=1.2, handletextpad=0.4)

    ax = axs[0, 2]
    ax.plot(q_vals[valid], hq[valid], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    ax.axhline(0.5, color="#aaaaaa", lw=0.9, ls="--", alpha=0.9, label="$H=0.5$")
    ax.axhline(1.0, color=PLT_STYLE["line_alt"], lw=0.9, ls=":", alpha=0.9,
               label="$H=1.0$")
    q2_idx = np.argmin(np.abs(q_vals - 2.0))
    if np.isfinite(hq[q2_idx]):
        ax.plot(q_vals[q2_idx], hq[q2_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=f"$H(q{{=}}2)={H2:.3f}$")
    ax.set_xlabel("$q$",    fontsize=10)
    ax.set_ylabel("$H(q)$", fontsize=10)
    ax.set_title("(c) Generalised Hurst exponent $H(q)$", fontsize=10)
    _legend(ax, fontsize=8)
    ax.text(0.03, 0.05,
            fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=8,
            color=PLT_STYLE["line_grn"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.85))

    ax = axs[1, 0]
    ax.plot(q_vals[valid], tau_q[valid], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    ax.axhline(0, color="#aaaaaa", lw=0.8, ls="--", alpha=0.8)
    ax.set_xlabel("$q$",        fontsize=10)
    ax.set_ylabel(r"$\tau(q)$", fontsize=10)
    ax.set_title(r"(d) Scaling exponent $\tau(q)$", fontsize=10)

    ax = axs[1, 1]
    ax.plot(q_vals[valid_s], alpha[valid_s], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    q0_idx = np.argmin(np.abs(q_vals))
    if np.isfinite(alpha[q0_idx]):
        ax.plot(q_vals[q0_idx], alpha[q0_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=fr"$\alpha_0={alpha0:.3f}$")
    ax.set_xlabel("$q$",          fontsize=10)
    ax.set_ylabel(r"$\alpha(q)$", fontsize=10)
    ax.set_title(r"(e) Singularity strength $\alpha(q)$", fontsize=10)
    _legend(ax, fontsize=8)

    ax = axs[1, 2]
    ax.plot(alpha[valid_s], D_alpha[valid_s], "-",
            color=PLT_STYLE["line_main"], linewidth=2.0)
    if np.isfinite(alpha[q0_idx]) and np.isfinite(D_alpha[q0_idx]):
        ax.plot(alpha[q0_idx], D_alpha[q0_idx], "o",
                color=PLT_STYLE["line_alt"], markersize=7, zorder=6,
                label=f"$q=0$  ($\\alpha_0={alpha0:.3f}$)")
    if valid_s.sum() >= 2:
        peak_idx   = np.nanargmax(D_alpha[valid_s])
        alpha_peak = alpha[valid_s][peak_idx]
        f_peak     = D_alpha[valid_s][peak_idx]
        ax.plot(alpha_peak, f_peak, "^",
                color=PLT_STYLE["line_grn"], markersize=9, zorder=7,
                label=fr"$D_H = \max f(\alpha) = {D_H_spectral:.4f}$")
        ax.annotate(fr"$D_H={D_H_spectral:.4f}$",
                    xy=(alpha_peak, f_peak),
                    xytext=(alpha_peak + 0.02, f_peak - 0.06),
                    fontsize=8, color=PLT_STYLE["line_grn"],
                    arrowprops=dict(arrowstyle="->", color=PLT_STYLE["line_grn"],
                                   lw=0.9))
        a_min  = alpha[valid_s].min()
        a_max  = alpha[valid_s].max()
        y_span = D_alpha[valid_s].min() - 0.05
        ax.annotate("", xy=(a_max, y_span), xytext=(a_min, y_span),
                    arrowprops=dict(arrowstyle="<->", color="#888888", lw=1.2))
        ax.text((a_min + a_max) / 2, y_span - 0.04,
                fr"$\Delta\alpha={delta_alpha:.3f}$",
                ha="center", fontsize=8, color=PLT_STYLE["text"])
    ax.set_xlabel(r"$\alpha$",    fontsize=10)
    ax.set_ylabel(r"$f(\alpha)$", fontsize=10)
    ax.set_title(r"(f) Multifractal spectrum $f(\alpha)$", fontsize=9)
    _legend(ax, fontsize=8)
    ax.text(0.97, 0.30,
            fr"$\lambda_{{MF}} = {lam_mf:.5f}$"
            "\n"
            fr"$= H_2\cdot\sigma\cdot(1+\Delta\alpha)$"
            "\n"
            fr"$D_H = {D_H_spectral:.4f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            color=PLT_STYLE["text"],
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor=PLT_STYLE["spine"], alpha=0.9))

    plt.tight_layout()
    fname = os.path.join(mfd_dir, f"{ticker.replace('=','_')}_mfdfa.png")
    plt.savefig(fname, dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Ticker: {ticker}")

    char_dir, mfd_dir, second_dir = get_dirs(ticker)
    logp, ret = get_data(ticker)

    if ret is None:
        print("  Skipping — insufficient data.")
        continue

    N = len(ret)
    print(f"  N = {N} returns")

    # MF-DFA
    scale_max_eff = min(SCALE_MAX, N // 4)
    scale_min_eff = max(SCALE_MIN, POLY_ORD + 2)
    print(f"  → MF-DFA  (scales [{scale_min_eff}, {scale_max_eff}]) …")

    scales, Fqs, hq = mfdfa(
        ret, Q_HURST,
        scale_min=scale_min_eff, scale_max=scale_max_eff,
        n_scales=N_SCALES, poly_ord=POLY_ORD, overlap=OVERLAP,
    )
    tau_q, alpha, D_alpha = multifractal_spectrum(Q_HURST, hq)

    lam_dict = lyapunov_mf(ret, Q_HURST, hq, alpha, D_alpha)
    print(f"  λ_MF        = {lam_dict['lam_mf']:.5f}")
    print(f"  D_H         = {lam_dict['D_H_spectral']:.5f}")

    # First-order characteristic analysis
    print("  → First-order characteristic analysis …")
    plot_characteristic(ticker, ret, lam_dict, char_dir)
    
    # MF-DFA plot
    plot_mfdfa(ticker, ret, scales, Fqs, hq, Q_HURST,
               tau_q, alpha, D_alpha, lam_dict, mfd_dir)
    
    # SECOND-ORDER characteristic analysis (NEW)
    # This applies the same transform to the characteristic curves themselves!
    analyze_second_order(ticker, ret, char_dir, second_dir)

print(f"\n{'='*60}")
print(f"Done.  All results saved in:  {OUT_DIR}/")


Ticker: AAPL
  N = 4062 returns
  → MF-DFA  (scales [20, 800]) …
  λ_MF        = 0.01416
  D_H         = 1.01576
  → First-order characteristic analysis …
    saved → fractal_structure_results\AAPL\characteristic_curves\AAPL_characteristic.png
    saved → fractal_structure_results\AAPL\mfdfa\AAPL_mfdfa.png

  → SECOND-ORDER CHARACTERISTIC ANALYSIS
    (applying the same transform to the characteristic curves themselves)
    Computing first-order characteristic curves...
      q=0: curve with 2000 points
      q=1: curve with 2000 points
      q=5: curve with 2000 points
      q=10: curve with 2000 points
      q=20: curve with 2000 points
      q=60: curve with 2000 points
      q=120: curve with 2000 points
      q=200: curve with 2000 points
      q=300: curve with 2000 points
      q=20, q2=0: D_H(G) = 1.0000
      q=20, q2=1: D_H(G) = 1.0000
      q=20, q2=2: D_H(G) = 1.0000
      q=20, q2=5: D_H(G) = 1.0000
      q=20, q2=10: D_H(G) = 1.0000
      q=20, q2=20: D_H(G) = 1.0000
   